# 10 · Joint model plus richer five-minute paths

Controlled addition of twelve dimensionless, previous-close five-minute path inputs to the completed joint30 model. All thirty existing inputs, forecast rows, targets, labels, random inner splits, seeds, tree parameters and portfolios are preserved. No exact duplicate columns at float32 precision and no target-based feature selection.

This notebook inspects frozen releases only. It never builds missing features or starts training. No one-minute data are used.

In [ ]:
from pathlib import Path
import json,os,sys
PROJECT_DIR=next(p for p in [Path.cwd(),*Path.cwd().parents] if (p/"scripts/joint_features.py").exists())
sys.path.insert(0,str(PROJECT_DIR/"scripts"))
RUN_DIR=PROJECT_DIR/"data/models/10_xgb_ret_rich_path100_v1"
BASELINE_DIR=PROJECT_DIR/"data/models/09_xgb_ret_joint100_v1"
FEATURE_DIR=PROJECT_DIR/"data/features/10_rich_5min_path_v1"
REPORT_DIR=PROJECT_DIR/"data/reports/10_train_xgb_ret_rich_path"
REPORT_DIR.mkdir(parents=True,exist_ok=True)
os.environ.setdefault("MPLCONFIGDIR",str(REPORT_DIR/".matplotlib"))
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display,Image,Markdown
from ret_backtest import performance
from rich_5min_features import MODEL_FEATURES
RUN_TRAINING=False  # This notebook inspects only; the explicit CLI starts training.
manifest=json.loads((FEATURE_DIR/"MANIFEST.json").read_text())
assert manifest["model_features"]==MODEL_FEATURES and len(set(MODEL_FEATURES))==42
assert manifest["summary"]["cohort_labels_outcomes_identical"]
display(pd.DataFrame.from_dict(manifest["summary"],orient="index",columns=["feature_release"]))
display(pd.DataFrame({"feature":MODEL_FEATURES}))
if (RUN_DIR/"CONFIG.json").exists():
    config=json.loads((RUN_DIR/"CONFIG.json").read_text())
    baseline_config=json.loads((BASELINE_DIR/"CONFIG.json").read_text())
    for key in ["parameters","members_per_quarter","total_quarters","train_months","test_months","early_stopping_patience","n_per_side","gross_exposure","inner_split","member_prediction","averaging"]:
        assert config[key]==baseline_config[key],key
    assert config["max_rounds"] is None and config["memory_limit_gib"] is None and config["hard_stop_utc"] is None
    assert config["minimum_available_memory_gib"]==0


In [ ]:
distribution=pd.read_csv(FEATURE_DIR/"feature_distribution.csv")
display(distribution[distribution.feature.str.startswith("path_")])
assert manifest["one_minute_data_used"] is False
assert manifest["summary"]["source_features_bitwise_preserved"]
assert not manifest["summary"]["exact_duplicate_features_removed"]
display(pd.DataFrame.from_dict(manifest["summary"]["quality"],orient="index",columns=["stock_days"]))

## Feature construction and fitting

Intraday log increments start at first-bar open to close, then use adjacent scheduled closes; no overnight move enters the new path block. Added inputs measure relative RMS volatility, downside variance share, signed efficiency, closing-hour variance concentration, late versus early return, scaled maximum drawdown of the observed close path, time above cumulative VWAP, adjacent-return correlation, high/low timing, return/activity correlation, profile-relative closing activity and its signed balance.

All additions are known by the previous session close. Activity uses raw VWAP × shares relative to that stock’s same five-minute slot over the prior ten scheduled sessions, excluding today. Structural early-close slots are handled separately. Undefined new values stay native NaN: no rows or prices are filled or removed. Row-level flags and missing masks remain in the feature release.

Forty outer quarters, 100 fresh models per quarter, 12 training months and 3 forecast months. Ten one-thread workers. Identical tree settings, deterministic seed and authorized random 80/20 inner-split schedule. Stop after 30 non-improving validation rounds; no round, RAM, headroom or runtime ceilings.

The existing trade-day opening-gap assumption remains optimistic: the opening print is used to rank and fill at that same price, while its bar is delivered later. These repeatedly viewed 2017–2026 periods are development evidence, not a fresh final holdout.

In [ ]:
status=json.loads((RUN_DIR/"RUN_STATUS.json").read_text()) if (RUN_DIR/"RUN_STATUS.json").exists() else {"status":"prepared"}
display(pd.DataFrame.from_dict(status,orient="index",columns=["run_status"]))
COMPLETE=(RUN_DIR/"RESULTS.json").exists() and status.get("status")=="complete"
if not COMPLETE: display(Markdown("Training is scheduled/running. Full-ensemble performance is withheld until all 4,000 models finish."))

## Controlled comparison against joint30

Both models and N=2/3 use identical stock-day forecasts. Portfolio statistics use the same observed dates across all four portfolios, keep cash sessions and explicitly export omitted dates. Missing selected exits stay unknown and are never replaced. Curves on conditional dates do not establish a complete executable path. Cost scenarios are sensitivities, without measured slippage or borrow.

The following audit checks every member’s split and forecast-row fingerprints and every quarter’s RET payoff vector against joint30.

In [ ]:
if COMPLETE:
    books=[]
    for label,folder in [("joint30",BASELINE_DIR),("path42",RUN_DIR)]:
        b=pd.read_parquet(folder/"portfolio_daily.parquet");b["feature_set"]=label;books.append(b)
    books=pd.concat(books,ignore_index=True)
    wide=books.pivot(index="trade_date",columns=["feature_set","n_per_side"],values="gross_return")
    common=wide.notna().all(axis=1);dates=common[common].index
    records=[]
    for (label,n),book in books.groupby(["feature_set","n_per_side"]):
        observed=book.set_index("trade_date").loc[dates]
        for cost in [0,1,5,10]:
            records.append(dict(feature_set=label,n_per_side=n,cost_bps_per_side=cost,
                calendar_sessions=len(book),omitted_dates=int((~common).sum()),
                metric_scope="four_way_common_observed_dates_conditional",
                **performance(observed.gross_return-observed.trade_notional*cost/10000)))
    comparison=pd.DataFrame(records);comparison.to_parquet(REPORT_DIR/"common_date_comparison.parquet",index=False)
    common.to_frame("included").to_parquet(REPORT_DIR/"common_date_mask.parquet")
    display(comparison)
    fig,axes=plt.subplots(1,2,figsize=(12,4))
    for ax,cost in zip(axes,[0,1]):
        for (label,n),book in books.groupby(["feature_set","n_per_side"]):
            b=book.set_index("trade_date").loc[dates]
            ax.plot(dates,(1+b.gross_return-b.trade_notional*cost/10000).cumprod(),label=f"{label}, N={n}")
        ax.set_title(f"Conditional equity, {cost} bp/side");ax.legend(fontsize=8);ax.grid(alpha=.2)
    fig.tight_layout();fig.savefig(REPORT_DIR/"path_comparison.png",dpi=150);plt.show()

In [ ]:
if COMPLETE:
    fingerprints=["split_seed","booster_seed","split_train_index_sha256","split_validation_index_sha256","prediction_row_key_sha256"]
    members=0
    for folder in sorted((RUN_DIR/"folds").iterdir()):
        if not folder.is_dir():continue
        current=json.loads((folder/"FOLD.json").read_text())
        previous=json.loads((BASELINE_DIR/"folds"/folder.name/"FOLD.json").read_text())
        np.testing.assert_allclose(current["payoff_vector_class_1_to_10"],previous["payoff_vector_class_1_to_10"],atol=1e-12,rtol=0)
        paths=list((folder/"members").glob("member_*.json"));assert len(paths)==100
        for path in paths:
            a=json.loads(path.read_text());b=json.loads((BASELINE_DIR/"folds"/folder.name/"members"/path.name).read_text())
            assert all(a[k]==b[k] for k in fingerprints)
        members+=len(paths)
    assert members==4000
    display(pd.DataFrame({label:json.loads((folder/"RESULTS.json").read_text())["classification"] for label,folder in [("joint30",BASELINE_DIR),("path42",RUN_DIR)]}))
    (REPORT_DIR/"COMPARISON_AUDIT.json").write_text(json.dumps(dict(members=members,split_row_fingerprints_match=True,payoffs_match=True,cohort_labels_outcomes_unchanged=True),indent=2))

## Importance, explained variance and calibration

Gain shares describe in-training classifier splits, not explained return
variance or causal effects. EV and R² use saved ensemble RET forecasts on exact
matched observed stock-days. Negative scores remain visible. Different
normalizations can share correlated information despite having no identical
columns. No fresh final holdout or delayed-execution evidence is claimed.

In [ ]:
if COMPLETE:
    from analyze_volume_overnight_results import analyze
    diagnostics=analyze(PROJECT_DIR,experiment="path42")
    display(pd.read_parquet(diagnostics/"feature_importance.parquet"))
    display(pd.read_parquet(diagnostics/"explained_variance.parquet"))
    for filename in ["feature_importance.png","feature_importance_by_quarter.png","out_of_sample_explained_variance.png","explained_variance_by_year.png","out_of_sample_return_calibration.png"]:
        display(Image(filename=str(diagnostics/filename)))
    (REPORT_DIR/"SUMMARY.json").write_text(json.dumps(dict(status="complete",models=4000,common_observed_dates=len(dates),omitted_dates=int((~common).sum()),comparison=comparison.to_dict("records")),indent=2,default=str))